# Sales Insight:

### Pipeline de Análise e Visualização de Dados de Vendas

<br>

Importando bibliotecas iniciais

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re
import matplotlib.pyplot as plt 
import seaborn as sns 
import os 
import json

Importando o dataset e visualização das 6 primeiras linhas

In [3]:
df_bruto = pd.read_csv('./vendas.csv', sep=';')

# Verificando as 6 primeiras linhas do dataset `vendas.csv`

df_bruto.head(6)

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-05-20,Cliente_035,Mouse,Periféricos,Sudeste,2.0,102.90
1,2,2024-02-17,Cliente_042,Teclado,Periféricos,Norte,7.0,214.88
2,3,2024-05-22,Cliente_022,Monitor,Computadores,Nordeste,4.0,NaN
3,4,2024-06-21,Cliente_017,Smartphone,Celulares,Sudeste,4.0,2501.76
4,5,2024-07-12,Cliente_024,Mouse,Periféricos,Norte,8.0,121.30
5,6,2024-04-26,Cliente_025,Smartphone,Celulares,Nordeste,2.0,1900.24


Inspeção Inicial do Dataset

In [4]:

def inspecionar_dados(df_bruto): 
    """Exibe informações básicas do DataFrame.""" 
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===") 
    print(f"Shape: {df_bruto.shape}") 
    print(f"\nColunas: {list(df_bruto.columns)}") 
    print(f"\nTipos de dados:\n{df_bruto.dtypes}") 
    print(f"\nValores nulos por coluna:\n{df_bruto.isnull().sum()}") 
    print(f"\nPrimeiros registros:\n{df_bruto.head()}") 
    print(f"\nEstatísticas descritivas:\n{df_bruto.describe()}") 

In [5]:
dados_brutos = inspecionar_dados(df_bruto)


=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade         6
preco_unitario    10
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente     produto     categoria    regiao  \
0         1  2024-05-20  Cliente_035       Mouse   Periféricos   Sudeste   
1         2  2024-02-17  Cliente_042     Teclado   Periféricos     Norte   
2         3  2024-05-22  Cliente_022     Monitor  Computadores  Nordeste   
3         4  2024-06-21  Cliente_017  Smartphone     Celular

In [6]:

# Após rápida avaliação do dataset bruto, notou-se a presença de `"DATA INVÁLIDA"` em `data_venda`

df_bruto[df_bruto['data_venda'] == 'DATA INVÁLIDA']

,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
56,57,DATA INVÁLIDA,Cliente_049,Monitor,Computadores,Norte,1.0,1094.46
91,92,DATA INVÁLIDA,Cliente_027,Notebook,Computadores,Sudeste,1.0,3996.80
94,95,DATA INVÁLIDA,Cliente_047,Headset,Periféricos,Nordeste,2.0,326.51
167,168,DATA INVÁLIDA,Cliente_023,Teclado,Periféricos,Sul,9.0,213.73


In [7]:

total = len(df_bruto)
invalidas = (df_bruto['data_venda'] == 'DATA INVÁLIDA').sum()
percentual = (invalidas / total) * 100

print(f"Datas inválidas: {invalidas} de {total} ({percentual:.2f}%)")

Datas inválidas: 4 de 200 (2.00%)


<br>

<u>Análise inicial do Dataset:</u>

1. 200 registros, 8 colunas
2. 6 nulos em `quantidade` e 10 nulos em `preco_unitario` — serão tratados na limpeza
3. `data_venda` como `str` — datas inválidas serão identificadas e tratadas antes da conversão para `datetime`
4. `quantidade` como `float64` — pode ser convertido para `int` após tratamento dos nulos

**Ponto de atenção:** `data_venda` aparece como `str` e `isnull().sum()` retorna 0 para ela — isso significa que linhas com `"DATA INVÁLIDA"` não são detectadas como nulas, pois são strings válidas para o pandas.

<br>

Limpeza e tratamento dos dados

In [8]:
df = df_bruto.copy()

def limpar_dados(df): 
    """ 
    Limpa e trata o DataFrame de vendas. 
    Retorna o DataFrame limpo e um relatório de limpeza. 
    """ 

    n_inicial = len(df) 
    relatorio = {} 
 
    # 1. Remover espaços extras em colunas de texto 
    colunas_texto = df.select_dtypes(include=["object", "string"]).columns
    for col in colunas_texto: 
        df[col] = df[col].str.strip()

    # 2. Usar regex para limpar caracteres especiais em 'cliente'
    if "cliente" in df.columns:
        df["cliente"] = df["cliente"].apply(lambda x: re.sub(r"[^A-Za-z0-9_]", "", str(x)))
 
    # 3. Converter data e remover datas inválidas 
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce") 
    n_datas_invalidas = df["data_venda"].isnull().sum() 
    df = df.dropna(subset=["data_venda"]) 
    relatorio["datas_invalidas_removidas"] = n_datas_invalidas 
 
    # 4. Remover linhas com quantidade ou preço nulos 
    n_antes = len(df) 
    df = df.dropna(subset=["quantidade", "preco_unitario"]) 
    relatorio["linhas_nulas_removidas"] = n_antes - len(df) 
 
    # 5. Garantir tipos numéricos corretos 
    df["quantidade"] = df["quantidade"].astype(int) 
    df["preco_unitario"] = df["preco_unitario"].astype(float) 
 
    n_final = len(df) 
    relatorio["registros_iniciais"] = n_inicial 
    relatorio["registros_finais"] = n_final 
    relatorio["registros_removidos_total"] = n_inicial - n_final
    relatorio["percentual_removido"] = round((n_inicial - n_final) / n_inicial * 100, 2) # Informação adicionada
 
    print("\n=== RELATÓRIO DE LIMPEZA ===") 
    for chave, valor in relatorio.items(): 
        print(f" {chave}: {valor}")

    return df, relatorio

In [9]:
# Chamando a função para geração do relatório da limpeza

df_limpo, relatorio = limpar_dados(df_bruto)

df_limpo.head(6)


=== RELATÓRIO DE LIMPEZA ===
 datas_invalidas_removidas: 4
 linhas_nulas_removidas: 16
 registros_iniciais: 200
 registros_finais: 180
 registros_removidos_total: 20
 percentual_removido: 10.0


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario
0,1,2024-05-20,Cliente_035,Mouse,Periféricos,Sudeste,2,102.90
1,2,2024-02-17,Cliente_042,Teclado,Periféricos,Norte,7,214.88
3,4,2024-06-21,Cliente_017,Smartphone,Celulares,Sudeste,4,2501.76
4,5,2024-07-12,Cliente_024,Mouse,Periféricos,Norte,8,121.30
5,6,2024-04-26,Cliente_025,Smartphone,Celulares,Nordeste,2,1900.24
6,7,2024-06-30,Cliente_039,Monitor,Computadores,Sul,6,1078.56


<br>

<u>Avaliação pós limpeza do dataset:</u>

1. O dataset foi reduzido de 200 para 180 linhas após a remoção de nulos e das strings `"DATA INVÁLIDA"`. A perda total foi de 10%.
2. A coluna `data_venda` foi convertida com sucesso para `datetime64` e `quantidade` para `int`.
3. Espaços extras foram removidos e a coluna `cliente` foi normalizada via regex, restando apenas caracteres alfanuméricos.
4. O dataset está livre de nulos e com a tipagem correta, totalmente pronto para a engenharia de atributos e modelagem preditiva.

<br>


Criação de colunas derivadas com transformações

In [12]:
def aplicar_transformacao(serie, funcao_transformacao):
    """
    Aplica uma função recebida como parâmetro a uma Series.
    Demonstra o uso de função de ordem superior.
    """
    return serie.apply(funcao_transformacao)

In [13]:
def criar_colunas_derivadas(df_limpo):
    """Cria colunas calculadas e derivadas a partir do dataset limpo.""" 
 
    # Receita total por linha de venda 
    df_limpo["receita_total"] = df_limpo["quantidade"] * df_limpo["preco_unitario"] 
 
    # Extração de componentes de data 
    df_limpo["mes"] = df_limpo["data_venda"].dt.month 
    df_limpo["mes_nome"] = df_limpo["data_venda"].dt.strftime("%B")  # nome do mês 
    df_limpo["trimestre"] = df_limpo["data_venda"].dt.quarter.apply(lambda q: f"Q{q}") 
    df_limpo["ano"] = df_limpo["data_venda"].dt.year 
 
    # Classificação da receita por item com numpy.select (transformação condicional vetorizada) 
    condicoes = [ 
        df_limpo["receita_total"] < 500, 
        (df_limpo["receita_total"] >= 500) & (df_limpo["receita_total"] < 5000), 
        df_limpo["receita_total"] >= 5000 
        ] 
    classificacoes = ["Baixo Valor", "Médio Valor", "Alto Valor"] 
    df_limpo["faixa_receita_item"] = np.select(condicoes, classificacoes, default="Não Classificado")

    print("\n=== COLUNAS DERIVADAS CRIADAS ===") 
    print(df_limpo[["data_venda", "receita_total", "mes", "trimestre", "faixa_receita_item"]].head()) 
    return df_limpo

In [11]:
# Chamando a função de colunas derivadas garantindo uma cópia limpa do dataframe

df_transformado = criar_colunas_derivadas(df_limpo.copy())

df_transformado.head(5)


=== COLUNAS DERIVADAS CRIADAS ===
  data_venda  receita_total  mes trimestre faixa_receita_item
0 2024-05-20         205.80    5        Q2        Baixo Valor
1 2024-02-17        1504.16    2        Q1        Médio Valor
3 2024-06-21       10007.04    6        Q2         Alto Valor
4 2024-07-12         970.40    7        Q3        Médio Valor
5 2024-04-26        3800.48    4        Q2        Médio Valor


,id_venda,data_venda,cliente,produto,categoria,regiao,quantidade,preco_unitario,receita_total,mes,mes_nome,trimestre,ano,faixa_receita_item
0,1,2024-05-20,Cliente_035,Mouse,Periféricos,Sudeste,2,102.90,205.80,5,May,Q2,2024,Baixo Valor
1,2,2024-02-17,Cliente_042,Teclado,Periféricos,Norte,7,214.88,1504.16,2,February,Q1,2024,Médio Valor
3,4,2024-06-21,Cliente_017,Smartphone,Celulares,Sudeste,4,2501.76,10007.04,6,June,Q2,2024,Alto Valor
4,5,2024-07-12,Cliente_024,Mouse,Periféricos,Norte,8,121.30,970.40,7,July,Q3,2024,Médio Valor
5,6,2024-04-26,Cliente_025,Smartphone,Celulares,Nordeste,2,1900.24,3800.48,4,April,Q2,2024,Médio Valor


<br>

<u>Engenharia de Recursos (Colunas Derivadas):</u>

1. `receita_total`: Criada a variável alvo calculada (`quantidade` × `preco_unitario`).
2. Decomposição Temporal: Extraídos `mes`, `mes_nome`, `trimestre` e `ano` a partir de `data_venda` para capturar padrões de sazonalidade.
3. Classificação do faturamento dos itens em três faixas de valor (*Baixo, Médio e Alto*) de forma vetorizada via `np.select`.

<br>

Cálculo de métricas agregadas

In [ ]:
# Vai que é tua, Luis

# Fiz ctrl+c, ctrl+v da edição que eu tinha subido no github

# Se puder, inclua a parte de função que recebe função
# Eu vi que vc incluiu em algum lugar da limpeza, mas não consegui achar

# Vlws!!

In [14]:
def calcular_metricas(df_limpo): 
    """Calcula e retorna métricas agregadas do dataset.""" 
    metricas = {} 
    # Receita por mês 
    por_mes = df_limpo.groupby("mes").agg( receita_total=("receita_total", "sum"), 
        quantidade=("quantidade", "sum"), 
        n_vendas=("id_venda", "count") 
    ).reset_index().sort_values("mes") 
    metricas["receita por mes"] = por_mes 
 
    # Top 5 produtos por receita 
    top_produtos = df_limpo.groupby("produto")["receita_total"].sum().sort_values(ascending=False).head(5).reset_index() 
    metricas["top 5 produtos por receita"] = top_produtos 
 
    # Receita por categoria 
    por_categoria = df_limpo.groupby("categoria")["receita_total"].sum().reset_index() 
    metricas["receita por categoria"] = por_categoria 
 
    # Receita por região 
    por_regiao = df_limpo.groupby("regiao").agg( 
        receita_total=("receita_total", "sum"), 
        media_ticket=("receita_total", "mean") 
    ).reset_index().sort_values("receita_total", ascending=False) 
    metricas["receita por regiao"] = por_regiao 
 
    # Exibição 
    for nome, tabela in metricas.items(): 
        print(f"\n=== {nome.upper().replace('_', ' ')} ===") 
        print(tabela.to_string(index=False)) 
 
    return metricas

In [15]:
def segmentar_clientes(df_limpo): 
    """Segmenta clientes pelo total gasto usando groupby e lambda.""" 
    clientes = df_limpo.groupby("cliente")["receita_total"].sum().reset_index() 
    clientes.columns = ["cliente", "total_gasto"] 

    # Classificação usando função lambda com condicionais 
    clientes["segmento"] = clientes["total_gasto"].apply( 
        lambda gasto: "Ouro" if gasto > 15000 
        else ("Prata" if gasto >= 5000 else "Bronze") 
        ) 
    clientes = clientes.sort_values("total_gasto", ascending=False)

    print("\n=== SEGMENTAÇÃO DE CLIENTES ===") 
    print(clientes.head(10).to_string(index=False)) 
    print(f"\nDistribuição de segmentos:\n{clientes['segmento'].value_counts()}") 

    return clientes

In [16]:
def calcular_estatisticas_numpy(df_limpo): 
    """Usa NumPy para calcular estatísticas sobre as receitas.""" 
    print("\n=== ESTATÍSTICAS COM NUMPY ===") 
 
    receitas = df_limpo["receita_total"].to_numpy()  # Converte para array NumPy 
 
    media = np.mean(receitas) 
    mediana = np.median(receitas) 
    desvio_padrao = np.std(receitas) 
    total = np.sum(receitas) 
    p25 = np.percentile(receitas, 25) 
    p75 = np.percentile(receitas, 75) 
 
    print(f"  Receita média por venda:    R$ {media:.2f}") 
    print(f"  Receita mediana por venda:  R$ {mediana:.2f}") 
    print(f"  Desvio padrão:              R$ {desvio_padrao:.2f}") 
    print(f"  Receita total:              R$ {total:.2f}") 
    print(f"  Percentil 25 (Q1):          R$ {p25:.2f}") 
    print(f"  Percentil 75 (Q3):          R$ {p75:.2f}") 
 
    # Broadcasting: normalizar receitas entre 0 e 1 
    receitas_normalizadas = (receitas - receitas.min()) / (receitas.max() - receitas.min()) 
    print(f"\n  Receitas normalizadas (primeiros 5): {receitas_normalizadas[:5].round(4)}") 
 
    # Operação vetorizada: identificar vendas acima da média sem loop 
    acima_da_media = receitas[receitas > media] 
    print(f"\n  Vendas acima da média: {len(acima_da_media)} de {len(receitas)}") 
 
    return { 
        "media": media, "mediana": mediana, 
        "desvio_padrao": desvio_padrao, "total": total 
    }